In [2]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.ensemble import StackingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report
import warnings
warnings.filterwarnings('ignore')

df_ml = pd.read_csv('Recruitment_ml_ready.csv')
X = df_ml.drop('was_placed', axis=1)
y = df_ml['was_placed']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                      random_state=42, stratify=y)
# XGBoost
xgb = XGBClassifier(n_estimators=200, learning_rate=0.05,
                     max_depth=4, random_state=42,
                     eval_metric='logloss', verbosity=0)
xgb.fit(X_train, y_train)

# Stacking ensemble
estimators = [
    ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
    ('xgb', XGBClassifier(n_estimators=100, random_state=42,
                           eval_metric='logloss', verbosity=0))
]
stack = StackingClassifier(estimators=estimators,
                            final_estimator=LogisticRegression(),
                            cv=5)
stack.fit(X_train, y_train)

# Evaluate all three
for name, model in [('XGBoost', xgb), ('Stacking', stack)]:
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:,1]
    print(f"\n=== {name} ===")
    print(classification_report(y_test, y_pred, target_names=['Not Placed','Placed']))
    print(f"ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}")


=== XGBoost ===
              precision    recall  f1-score   support

  Not Placed       0.17      0.04      0.06        26
      Placed       0.73      0.93      0.82        74

    accuracy                           0.70       100
   macro avg       0.45      0.49      0.44       100
weighted avg       0.59      0.70      0.62       100

ROC-AUC: 0.5525

=== Stacking ===
              precision    recall  f1-score   support

  Not Placed       0.00      0.00      0.00        26
      Placed       0.74      1.00      0.85        74

    accuracy                           0.74       100
   macro avg       0.37      0.50      0.43       100
weighted avg       0.55      0.74      0.63       100

ROC-AUC: 0.4475
